# Tackling Mode Collapse in GANs: DCGAN vs WGAN-GP
### Dual GPU (T4x2) | Mixed Precision | Pokemon Sprites + Anime Faces
---

## Cell 1 - Install Dependencies

In [1]:
!pip install -q kagglehub torchvision pytorch-fid gradio

## Cell 2 - Imports

In [2]:
import os, gc, json, random, shutil, zipfile, subprocess, warnings
import numpy as np
from pathlib import Path
from PIL import Image
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')  # no GUI - prevents kernel crash on Kaggle
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from torch.amp import GradScaler, autocast
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
print('GPUs   :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}]', torch.cuda.get_device_name(i),
          f'  {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA   : True
GPUs   : 1
  [0] Tesla T4   15.6 GB


## Cell 3 - Global Configuration

In [3]:
DATASET_CHOICE = 'anime'   # 'anime' or 'pokemon'

IMAGE_SIZE  = 64
BATCH_SIZE  = 64
NZ          = 100
NGF         = 64
NDF         = 64
NC          = 3

DCGAN_EPOCHS = 17
WGAN_EPOCHS  = 17
LR           = 0.0002
BETA1        = 0.5
BETA2        = 0.999
N_CRITIC     = 5
LAMBDA_GP    = 10

SHOW_EVERY   = 5   # show generated images every N epochs
CKPT_EVERY   = 5   # save checkpoint every N epochs

CKPT_DIR   = '/kaggle/working/checkpoints'
SAMPLE_DIR = '/kaggle/working/samples'
for d in [CKPT_DIR, f'{SAMPLE_DIR}/dcgan', f'{SAMPLE_DIR}/wgan']:
    os.makedirs(d, exist_ok=True)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP   = torch.cuda.is_available()
MULTI_GPU = torch.cuda.device_count() > 1

print(f'Device     : {device}')
print(f'Multi-GPU  : {MULTI_GPU}')
print(f'AMP        : {USE_AMP}')
print(f'Epochs     : DCGAN={DCGAN_EPOCHS}  WGAN-GP={WGAN_EPOCHS}')
print(f'Show every {SHOW_EVERY} epochs | Checkpoint every {CKPT_EVERY} epochs')

Device     : cuda
Multi-GPU  : False
AMP        : True
Epochs     : DCGAN=17  WGAN-GP=17
Show every 5 epochs | Checkpoint every 5 epochs


## Cell 4 - Download Datasets

In [4]:
import kagglehub

print('Downloading Pokemon Sprites...')
pokemon_path = kagglehub.dataset_download('jackemartin/pokemon-sprites')
print('  ->', pokemon_path)

print('Downloading Anime Faces...')
anime_path = kagglehub.dataset_download('soumikrakshit/anime-faces')
print('  ->', anime_path)

100%|██████████| 71.2M/71.2M [00:00<00:00, 110MB/s]

Extracting files...


  -> /root/.cache/kagglehub/datasets/jackemartin/pokemon-sprites/versions/1
Using Colab cache for faster access to the 'anime-faces' dataset.
  -> /kaggle/input/anime-faces


## Cell 5 - Explore Dataset Structure

In [5]:
def explore(root_path, max_show=20):
    root = Path(root_path)
    print(f'\n=== {root_path} ===')
    entries = sorted(root.iterdir())
    for e in entries[:max_show]:
        tag = '[DIR] ' if e.is_dir() else '[FILE]'
        n   = len(list(e.rglob('*'))) if e.is_dir() else ''
        print(f'  {tag} {e.name}  {n}')
    if len(entries) > max_show:
        print(f'  ... +{len(entries)-max_show} more')
    imgs = []
    for ext in ('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG'):
        imgs.extend(root.rglob(ext))
    imgs = [p for p in imgs if p.stat().st_size > 0]
    print(f'  Total images: {len(imgs)}')
    for p in imgs[:3]:
        print(f'    {p}')
    return [str(p) for p in imgs]

pokemon_imgs = explore(pokemon_path)
anime_imgs   = explore(anime_path)


=== /root/.cache/kagglehub/datasets/jackemartin/pokemon-sprites/versions/1 ===
  [DIR]  pokemon_images  33888
  Total images: 33887
    /root/.cache/kagglehub/datasets/jackemartin/pokemon-sprites/versions/1/pokemon_images/pokemondb.net/furfrou10.jpg
    /root/.cache/kagglehub/datasets/jackemartin/pokemon-sprites/versions/1/pokemon_images/pokemondb.net/squirtle39.jpg
    /root/.cache/kagglehub/datasets/jackemartin/pokemon-sprites/versions/1/pokemon_images/pokemondb.net/cobalion0.jpg

=== /kaggle/input/anime-faces ===
  [DIR]  data  43103
  Total images: 43102
    /kaggle/input/anime-faces/data/21130.png
    /kaggle/input/anime-faces/data/9273.png
    /kaggle/input/anime-faces/data/18966.png


## Cell 6 - Dataset and DataLoader

In [6]:
class ImageDataset(Dataset):
    EXTS = ('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG')

    def __init__(self, root_dir, size=64, subset=None):
        root = Path(root_dir)
        paths = []
        for ext in self.EXTS:
            paths.extend(root.rglob(ext))
        self.files = [str(p) for p in paths if p.stat().st_size > 0]
        if subset:
            random.shuffle(self.files)
            self.files = self.files[:subset]
        self.tf = transforms.Compose([
            transforms.Resize((size, size),
                interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        print(f'Dataset: {len(self.files)} images')

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        try:
            return self.tf(Image.open(self.files[idx]).convert('RGB'))
        except Exception:
            return self.__getitem__((idx + 1) % len(self.files))


ROOT = anime_path if DATASET_CHOICE == 'anime' else pokemon_path

ds = ImageDataset(ROOT, IMAGE_SIZE)
dataloader = DataLoader(
    ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
    persistent_workers=True   # avoids fork crash between training phases
)
print(f'Batches/epoch: {len(dataloader)}')

# Sanity check - show real images
real_batch = next(iter(dataloader))
print(f'Batch: {real_batch.shape}  min={real_batch.min():.2f}  max={real_batch.max():.2f}')

grid = make_grid(real_batch[:16], nrow=8, normalize=True, value_range=(-1,1))
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(grid.permute(1,2,0).numpy()); ax.axis('off')
ax.set_title('Real Training Samples')
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/real_samples.png', dpi=100)
plt.show(); plt.close('all'); gc.collect()
print('real_samples.png saved.')

Dataset: 43102 images
Batches/epoch: 673
Batch: torch.Size([64, 3, 64, 64])  min=-1.00  max=1.00
real_samples.png saved.


## Cell 7 - Model Architectures

In [7]:
def weights_init(m):
    cn = m.__class__.__name__
    if 'Conv' in cn:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cn:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class DCGANGenerator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz,    ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z): return self.main(z)


# NO Sigmoid - BCEWithLogitsLoss handles it (AMP-safe)
class DCGANDiscriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc,    ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1,    4, 1, 0, bias=False)
        )
    def forward(self, x): return self.main(x).view(-1)


class WGANGenerator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz,    ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z): return self.main(z)


# InstanceNorm, NO Sigmoid, NO BatchNorm - required for WGAN-GP
class WGANCritic(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc,    ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1,    4, 1, 0, bias=False)
        )
    def forward(self, x): return self.main(x).view(-1)


def unwrap(m): return m.module if hasattr(m, 'module') else m

netG_dc = DCGANGenerator(NZ, NGF, NC).to(device)
netD_dc = DCGANDiscriminator(NC, NDF).to(device)
netG_wg = WGANGenerator(NZ, NGF, NC).to(device)
netC_wg = WGANCritic(NC, NDF).to(device)

for net in [netG_dc, netD_dc, netG_wg, netC_wg]:
    net.apply(weights_init)

if MULTI_GPU:
    netG_dc = nn.DataParallel(netG_dc)
    netD_dc = nn.DataParallel(netD_dc)
    netG_wg = nn.DataParallel(netG_wg)
    netC_wg = nn.DataParallel(netC_wg)
    print('DataParallel on', torch.cuda.device_count(), 'GPUs')

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'DCGAN  G: {n_params(netG_dc):,}  D: {n_params(netD_dc):,}')
print(f'WGAN   G: {n_params(netG_wg):,}  C: {n_params(netC_wg):,}')

DCGAN  G: 3,576,704  D: 2,765,568
WGAN   G: 3,576,704  C: 2,765,568


## Cell 8 - Loss Functions, Optimizers, Scalers

In [8]:
# BCEWithLogitsLoss = Sigmoid + BCE fused, numerically stable, AMP-safe
criterion_bce = nn.BCEWithLogitsLoss()
REAL_LABEL = 0.9   # label smoothing prevents D from becoming overconfident
FAKE_LABEL = 0.0


def wasserstein_loss(score_real, score_fake):
    return score_fake.mean() - score_real.mean()


def gradient_penalty(critic, real, fake):
    """
    WGAN-GP: penalise ||grad C(x_hat)||_2 deviating from 1.
    Must run in float32 - do NOT wrap in autocast.
    """
    b     = real.size(0)
    alpha = torch.rand(b, 1, 1, 1, device=device).expand_as(real)
    # interpolated samples between real and fake
    interp = (alpha * real.float() + (1.0 - alpha) * fake.float()).requires_grad_(True)
    d_out  = critic(interp)
    grads  = torch.autograd.grad(
        outputs=d_out, inputs=interp,
        grad_outputs=torch.ones_like(d_out),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grads = grads.view(b, -1)
    return LAMBDA_GP * ((grads.norm(2, dim=1) - 1) ** 2).mean()


optG_dc = optim.Adam(netG_dc.parameters(), lr=LR, betas=(BETA1, BETA2))
optD_dc = optim.Adam(netD_dc.parameters(), lr=LR, betas=(BETA1, BETA2))
optG_wg = optim.Adam(netG_wg.parameters(), lr=LR, betas=(BETA1, BETA2))
optC_wg = optim.Adam(netC_wg.parameters(), lr=LR, betas=(BETA1, BETA2))

# Separate scalers for each network path
scaler_G_dc = GradScaler('cuda', enabled=USE_AMP)
scaler_D_dc = GradScaler('cuda', enabled=USE_AMP)
scaler_G_wg = GradScaler('cuda', enabled=USE_AMP)

# Fixed noise for apples-to-apples visualisation across epochs
fixed_noise = torch.randn(16, NZ, 1, 1, device=device)

print('Ready: loss functions / optimizers / scalers / fixed noise')

Ready: loss functions / optimizers / scalers / fixed noise


## Cell 9 - Visualisation Utility

In [9]:
def show_grid(imgs_tensor, title, save_path, nrow=8):
    """
    Denormalise [-1,1] -> [0,1], build grid, save PNG, display inline.
    Always closes figure + calls gc.collect() to prevent memory leak.
    """
    imgs = imgs_tensor.detach().cpu()
    grid = make_grid(imgs, nrow=nrow, normalize=True, value_range=(-1, 1), padding=2)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis('off')
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close('all')   # CRITICAL: free figure memory
    gc.collect()       # CRITICAL: free Python GC objects

print('show_grid utility ready.')

show_grid utility ready.


## Cell 10 - DCGAN Training (17 epochs, visualise every 5)

In [10]:
print('=' * 60)
print('  PHASE 1 : DCGAN TRAINING')
print('=' * 60)

dcgan_epoch_G, dcgan_epoch_D = [], []

netG_dc.train(); netD_dc.train()

for epoch in range(1, DCGAN_EPOCHS + 1):
    ep_g, ep_d = [], []
    loop = tqdm(dataloader, desc=f'DCGAN {epoch}/{DCGAN_EPOCHS}', leave=False)

    for real in loop:
        real = real.to(device, non_blocking=True)
        b    = real.size(0)

        # --- Train Discriminator ---
        netD_dc.zero_grad(set_to_none=True)
        lbl_real = torch.full((b,), REAL_LABEL, device=device)
        lbl_fake = torch.full((b,), FAKE_LABEL, device=device)

        with autocast('cuda', enabled=USE_AMP):
            loss_D_real = criterion_bce(netD_dc(real), lbl_real)

        noise = torch.randn(b, NZ, 1, 1, device=device)
        with autocast('cuda', enabled=USE_AMP):
            fake        = netG_dc(noise)
            loss_D_fake = criterion_bce(netD_dc(fake.detach()), lbl_fake)

        loss_D = loss_D_real + loss_D_fake
        scaler_D_dc.scale(loss_D).backward()
        scaler_D_dc.step(optD_dc)
        scaler_D_dc.update()

        # --- Train Generator ---
        netG_dc.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            loss_G = criterion_bce(netD_dc(fake), lbl_real)

        scaler_G_dc.scale(loss_G).backward()
        scaler_G_dc.step(optG_dc)
        scaler_G_dc.update()

        ep_g.append(loss_G.item())
        ep_d.append(loss_D.item())
        loop.set_postfix(G=f'{loss_G.item():.3f}', D=f'{loss_D.item():.3f}')

    avg_g, avg_d = np.mean(ep_g), np.mean(ep_d)
    dcgan_epoch_G.append(avg_g)
    dcgan_epoch_D.append(avg_d)
    print(f'DCGAN [{epoch:2d}/{DCGAN_EPOCHS}]  G={avg_g:.4f}  D={avg_d:.4f}')

    # Visualise every SHOW_EVERY epochs AND the final epoch
    if epoch % SHOW_EVERY == 0 or epoch == DCGAN_EPOCHS:
        netG_dc.eval()
        with torch.no_grad():
            samples = netG_dc(fixed_noise)
        netG_dc.train()
        show_grid(
            samples,
            title=f'DCGAN Generated Images - Epoch {epoch}',
            save_path=f'{SAMPLE_DIR}/dcgan/epoch_{epoch:03d}.png'
        )

    # Checkpoint every CKPT_EVERY epochs AND final epoch
    if epoch % CKPT_EVERY == 0 or epoch == DCGAN_EPOCHS:
        torch.save({
            'epoch': epoch,
            'netG' : unwrap(netG_dc).state_dict(),
            'netD' : unwrap(netD_dc).state_dict(),
        }, f'{CKPT_DIR}/dcgan_ep{epoch:03d}.pth')
        print(f'  checkpoint saved -> epoch {epoch}')

print('\nDCGAN Training Complete!')

  PHASE 1 : DCGAN TRAINING


DCGAN 1/17:   0%|          | 0/673 [00:00<?, ?it/s]

DCGAN [ 1/17]  G=5.4592  D=0.8472


DCGAN 2/17:   0%|          | 0/673 [00:00<?, ?it/s]

DCGAN [ 2/17]  G=4.9127  D=0.7161


...

DCGAN [15/17]  G=3.4622  D=0.5751
  checkpoint saved -> epoch 15


DCGAN 16/17:   0%|          | 0/673 [00:00<?, ?it/s]

DCGAN [16/17]  G=3.4585  D=0.5953


DCGAN 17/17:   0%|          | 0/673 [00:00<?, ?it/s]

DCGAN [17/17]  G=3.4600  D=0.6156
  checkpoint saved -> epoch 17

DCGAN Training Complete!


## Cell 11 - DCGAN Loss Curves

In [11]:
epochs = range(1, len(dcgan_epoch_G) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, dcgan_epoch_G, 'o-', color='steelblue', lw=2)
axes[0].set_title('DCGAN - Generator Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, dcgan_epoch_D, 's-', color='tomato', lw=2)
axes[1].set_title('DCGAN - Discriminator Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCE Loss')
axes[1].grid(True, alpha=0.3)

plt.suptitle('DCGAN Training Loss Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/dcgan_loss.png', dpi=120)
plt.show(); plt.close('all'); gc.collect()
print('DCGAN loss curves saved.')

DCGAN loss curves saved.


## Cell 12 - WGAN-GP Training (17 epochs, visualise every 5)

In [12]:
print('=' * 60)
print('  PHASE 2 : WGAN-GP TRAINING')
print('=' * 60)

wgan_epoch_G, wgan_epoch_C = [], []

netG_wg.train(); netC_wg.train()

for epoch in range(1, WGAN_EPOCHS + 1):
    ep_g, ep_c = [], []
    loop = tqdm(dataloader, desc=f'WGAN-GP {epoch}/{WGAN_EPOCHS}', leave=False)

    for real in loop:
        real = real.to(device, non_blocking=True)
        b    = real.size(0)

        # --- Train Critic N_CRITIC times ---
        for _ in range(N_CRITIC):
            netC_wg.zero_grad(set_to_none=True)
            noise  = torch.randn(b, NZ, 1, 1, device=device)
            # GP requires float32 - NO autocast wrapper here
            fake_c = netG_wg(noise).detach()
            s_real = netC_wg(real.float())
            s_fake = netC_wg(fake_c.float())
            gp     = gradient_penalty(netC_wg, real, fake_c)
            loss_C = wasserstein_loss(s_real, s_fake) + gp
            loss_C.backward()
            optC_wg.step()
            ep_c.append(loss_C.item())

        # --- Train Generator ---
        netG_wg.zero_grad(set_to_none=True)
        noise  = torch.randn(b, NZ, 1, 1, device=device)
        with autocast('cuda', enabled=USE_AMP):
            fake_g = netG_wg(noise)
            loss_G = -netC_wg(fake_g.float()).mean()
        scaler_G_wg.scale(loss_G).backward()
        scaler_G_wg.step(optG_wg)
        scaler_G_wg.update()

        ep_g.append(loss_G.item())
        loop.set_postfix(G=f'{loss_G.item():.3f}', C=f'{loss_C.item():.3f}')

    avg_g, avg_c = np.mean(ep_g), np.mean(ep_c)
    wgan_epoch_G.append(avg_g)
    wgan_epoch_C.append(avg_c)
    print(f'WGAN-GP [{epoch:2d}/{WGAN_EPOCHS}]  G={avg_g:.4f}  C={avg_c:.4f}')

    # Visualise every SHOW_EVERY epochs AND the final epoch
    if epoch % SHOW_EVERY == 0 or epoch == WGAN_EPOCHS:
        netG_wg.eval()
        with torch.no_grad():
            samples = netG_wg(fixed_noise)
        netG_wg.train()
        show_grid(
            samples,
            title=f'WGAN-GP Generated Images - Epoch {epoch}',
            save_path=f'{SAMPLE_DIR}/wgan/epoch_{epoch:03d}.png'
        )

    # Checkpoint every CKPT_EVERY epochs AND final epoch
    if epoch % CKPT_EVERY == 0 or epoch == WGAN_EPOCHS:
        torch.save({
            'epoch': epoch,
            'netG' : unwrap(netG_wg).state_dict(),
            'netC' : unwrap(netC_wg).state_dict(),
        }, f'{CKPT_DIR}/wgan_ep{epoch:03d}.pth')
        print(f'  checkpoint saved -> epoch {epoch}')

print('\nWGAN-GP Training Complete!')

  PHASE 2 : WGAN-GP TRAINING


WGAN-GP 1/17:   0%|          | 0/673 [00:00<?, ?it/s]

WGAN-GP [ 1/17]  G=21.1530  C=-18.7118


WGAN-GP 2/17:   0%|          | 0/673 [00:00<?, ?it/s]

WGAN-GP [ 2/17]  G=23.2044  C=-19.8228


...

WGAN-GP [15/17]  G=80.2459  C=-10.6352
  checkpoint saved -> epoch 15


WGAN-GP 16/17:   0%|          | 0/673 [00:00<?, ?it/s]

WGAN-GP [16/17]  G=86.9342  C=-10.6360


WGAN-GP 17/17:   0%|          | 0/673 [00:00<?, ?it/s]

WGAN-GP [17/17]  G=92.8491  C=-10.5891
  checkpoint saved -> epoch 17

WGAN-GP Training Complete!


## Cell 13 - WGAN-GP Loss Curves

In [13]:
epochs = range(1, len(wgan_epoch_G) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, wgan_epoch_G, 'o-', color='steelblue', lw=2)
axes[0].set_title('WGAN-GP - Generator Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Wasserstein Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, wgan_epoch_C, 's-', color='darkorange', lw=2)
axes[1].set_title('WGAN-GP - Critic Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Wasserstein + GP')
axes[1].grid(True, alpha=0.3)

plt.suptitle('WGAN-GP Training Loss Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/wgan_loss.png', dpi=120)
plt.show(); plt.close('all'); gc.collect()
print('WGAN-GP loss curves saved.')

WGAN-GP loss curves saved.


## Cell 14 - Combined Loss Comparison

In [14]:
ep = range(1, DCGAN_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ep, dcgan_epoch_G, 'o-', color='steelblue',  label='DCGAN G',   lw=2)
axes[0].plot(ep, wgan_epoch_G,  's-', color='darkorange', label='WGAN-GP G', lw=2)
axes[0].set_title('Generator Loss vs Epoch')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, dcgan_epoch_D, 'o-', color='tomato',  label='DCGAN D',     lw=2)
axes[1].plot(ep, wgan_epoch_C,  's-', color='purple',  label='WGAN-GP C',   lw=2)
axes[1].set_title('Discriminator/Critic Loss vs Epoch')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('DCGAN vs WGAN-GP Training Loss', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/combined_loss.png', dpi=120)
plt.show(); plt.close('all'); gc.collect()
print('Combined loss comparison saved.')

Combined loss comparison saved.


## Cell 15 - Visual Comparison (4 Panels)

In [15]:
netG_dc.eval(); netG_wg.eval()

torch.manual_seed(0)
cmp_noise = torch.randn(64, NZ, 1, 1, device=device)

with torch.no_grad():
    dc_raw = netG_dc(cmp_noise).cpu()
    wg_raw = netG_wg(cmp_noise).cpu()

dc_imgs = (dc_raw * 0.5 + 0.5).clamp(0, 1)
wg_imgs = (wg_raw * 0.5 + 0.5).clamp(0, 1)

# Panel 1 - 10-sample side-by-side rows
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
fig.suptitle('Panel 1 - Same Noise: DCGAN (top) vs WGAN-GP (bottom)',
             fontsize=12, fontweight='bold')
for j in range(10):
    axes[0, j].imshow(dc_imgs[j].permute(1,2,0).numpy())
    axes[0, j].axis('off')
    axes[1, j].imshow(wg_imgs[j].permute(1,2,0).numpy())
    axes[1, j].axis('off')
axes[0, 0].set_ylabel('DCGAN',   fontsize=9)
axes[1, 0].set_ylabel('WGAN-GP', fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/cmp_panel1_rows.png', dpi=130, bbox_inches='tight')
plt.show(); plt.close('all'); gc.collect()

# Panel 2 - 4x4 grids
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
ax1.imshow(make_grid(dc_imgs[:16], nrow=4, padding=3).permute(1,2,0).numpy())
ax1.axis('off'); ax1.set_title('DCGAN  4x4 Grid', fontweight='bold')
ax2.imshow(make_grid(wg_imgs[:16], nrow=4, padding=3).permute(1,2,0).numpy())
ax2.axis('off'); ax2.set_title('WGAN-GP  4x4 Grid', fontweight='bold')
plt.suptitle('Panel 2 - Image Quality Grids', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/cmp_panel2_grids.png', dpi=130, bbox_inches='tight')
plt.show(); plt.close('all'); gc.collect()

# Panel 3 - RGB histograms
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.suptitle('Panel 3 - Pixel Colour Distribution', fontsize=12, fontweight='bold')
for row, (imgs, lbl) in enumerate([(dc_imgs,'DCGAN'),(wg_imgs,'WGAN-GP')]):
    for ch, (col, cn) in enumerate(zip(['red','green','blue'],['R','G','B'])):
        axes[row, ch].hist(imgs[:,ch].numpy().flatten(), bins=100,
                           color=col, alpha=0.7, density=True)
        axes[row, ch].set_title(f'{lbl} - {cn}')
        axes[row, ch].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/cmp_panel3_histograms.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close('all'); gc.collect()

# Panel 4 - Diversity heatmap
dc_std = dc_imgs.numpy().std(axis=0).mean(axis=0)
wg_std = wg_imgs.numpy().std(axis=0).mean(axis=0)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Panel 4 - Diversity Heatmap (brighter = more variation across samples)',
             fontsize=11, fontweight='bold')
pairs = [('DCGAN', dc_std, 'hot', 0, 0.35),
         ('WGAN-GP', wg_std, 'hot', 0, 0.35),
         ('Diff (WGAN-GP - DCGAN)', wg_std - dc_std, 'RdBu_r', -0.15, 0.15)]
for ax, (title, data, cmap, vmin, vmax) in zip(axes, pairs):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/cmp_panel4_diversity.png', dpi=130, bbox_inches='tight')
plt.show(); plt.close('all'); gc.collect()

print('All 4 comparison panels saved.')

All 4 comparison panels saved.


## Cell 16 - Quantitative Metrics Table

In [16]:
torch.manual_seed(1)
eval_noise = torch.randn(256, NZ, 1, 1, device=device)
with torch.no_grad():
    dc_e = ((netG_dc(eval_noise) * 0.5 + 0.5).clamp(0,1)).cpu().numpy()
    wg_e = ((netG_wg(eval_noise) * 0.5 + 0.5).clamp(0,1)).cpu().numpy()

def cosine_div(arr, n=128):
    flat  = arr[:n].reshape(n, -1).astype(np.float32)
    norms = np.linalg.norm(flat, axis=1, keepdims=True) + 1e-8
    flat  = flat / norms
    sim   = flat @ flat.T
    return float(sim[~np.eye(n, dtype=bool)].mean())

rows = [
    ('G Loss (final)',              dcgan_epoch_G[-1],          wgan_epoch_G[-1],          'lower'),
    ('D/C Loss (final)',            dcgan_epoch_D[-1],          wgan_epoch_C[-1],          None),
    ('Pixel Std-Dev (high=diverse)',dc_e.std(axis=0).mean(),    wg_e.std(axis=0).mean(),   'higher'),
    ('Pixel Mean',                  dc_e.mean(),                wg_e.mean(),               None),
    ('Pixel Range',                 dc_e.max()-dc_e.min(),      wg_e.max()-wg_e.min(),     'higher'),
    ('Cosine Sim (low=diverse)',    cosine_div(dc_e),           cosine_div(wg_e),          'lower'),
]

print('\n' + '='*66)
print('       QUANTITATIVE COMPARISON: DCGAN vs WGAN-GP')
print('='*66)
print(f'  {"Metric":<32} {"DCGAN":>9} {"WGAN-GP":>9}  Winner')
print('-'*66)
for name, dv, wv, prefer in rows:
    if prefer == 'lower':   winner = 'DCGAN' if dv < wv else 'WGAN-GP'
    elif prefer == 'higher': winner = 'DCGAN' if dv > wv else 'WGAN-GP'
    else: winner = '---'
    print(f'  {name:<32} {dv:>9.4f} {wv:>9.4f}  {winner}')
print('='*66)


       QUANTITATIVE COMPARISON: DCGAN vs WGAN-GP
  Metric                               DCGAN   WGAN-GP  Winner
------------------------------------------------------------------
  G Loss (final)                      3.4600   92.8491  DCGAN
  D/C Loss (final)                    0.6156  -10.5891  ---
  Pixel Std-Dev (high=diverse)        0.2283    0.2335  WGAN-GP
  Pixel Mean                          0.6465    0.6460  ---
  Pixel Range                         1.0000    0.9999  DCGAN
  Cosine Sim (low=diverse)            0.8919    0.8881  WGAN-GP


## Cell 17 - FID Score

In [17]:
FID_N  = 1000
FID_R  = '/kaggle/working/fid_real'
FID_DC = '/kaggle/working/fid_dcgan'
FID_WG = '/kaggle/working/fid_wgan'
for d in [FID_R, FID_DC, FID_WG]:
    os.makedirs(d, exist_ok=True)

# Save real images
count = 0
for batch in dataloader:
    for img in batch:
        if count >= FID_N: break
        save_image((img * 0.5 + 0.5).clamp(0,1), f'{FID_R}/{count:05d}.png')
        count += 1
    if count >= FID_N: break
print(f'Real saved: {count}')

# Save DCGAN fakes
count = 0
with torch.no_grad():
    while count < FID_N:
        n  = min(BATCH_SIZE, FID_N - count)
        fk = ((netG_dc(torch.randn(n, NZ, 1, 1, device=device)) * 0.5 + 0.5).clamp(0,1)).cpu()
        for img in fk:
            save_image(img, f'{FID_DC}/{count:05d}.png'); count += 1
print(f'DCGAN fakes saved: {count}')

# Save WGAN fakes
count = 0
with torch.no_grad():
    while count < FID_N:
        n  = min(BATCH_SIZE, FID_N - count)
        fk = ((netG_wg(torch.randn(n, NZ, 1, 1, device=device)) * 0.5 + 0.5).clamp(0,1)).cpu()
        for img in fk:
            save_image(img, f'{FID_WG}/{count:05d}.png'); count += 1
print(f'WGAN-GP fakes saved: {count}')

dev_str = 'cuda' if torch.cuda.is_available() else 'cpu'

def run_fid(a, b, dev):
    r = subprocess.run(
        ['python', '-m', 'pytorch_fid', a, b, '--device', dev],
        capture_output=True, text=True
    )
    out = (r.stdout + r.stderr).strip()
    for line in out.split('\n'):
        if 'FID' in line: return line.strip()
    return out

print('\nComputing FID scores...')
print('DCGAN   FID:', run_fid(FID_R, FID_DC, dev_str))
print('WGAN-GP FID:', run_fid(FID_R, FID_WG, dev_str))
print('Note: Lower FID = better quality and diversity.')

Real saved: 1000
DCGAN fakes saved: 1000
WGAN-GP fakes saved: 1000

Computing FID scores...
DCGAN   FID: FID:  86.39230336897947
WGAN-GP FID: FID:  85.70395220896452
Note: Lower FID = better quality and diversity.


## Cell 18 - Save All Weights + ZIP for Download

In [18]:
# Unwrap DataParallel before saving so weights load on any device
torch.save(unwrap(netG_dc).state_dict(), f'{CKPT_DIR}/dcgan_generator_final.pth')
torch.save(unwrap(netD_dc).state_dict(), f'{CKPT_DIR}/dcgan_discriminator_final.pth')
torch.save(unwrap(netG_wg).state_dict(), f'{CKPT_DIR}/wgan_generator_final.pth')
torch.save(unwrap(netC_wg).state_dict(), f'{CKPT_DIR}/wgan_critic_final.pth')

# Full resumable checkpoints with optimizer states
torch.save({
    'epoch': DCGAN_EPOCHS, 'netG': unwrap(netG_dc).state_dict(),
    'netD': unwrap(netD_dc).state_dict(), 'optG': optG_dc.state_dict(),
    'optD': optD_dc.state_dict(), 'g_losses': dcgan_epoch_G, 'd_losses': dcgan_epoch_D,
}, f'{CKPT_DIR}/dcgan_full_checkpoint.pth')

torch.save({
    'epoch': WGAN_EPOCHS, 'netG': unwrap(netG_wg).state_dict(),
    'netC': unwrap(netC_wg).state_dict(), 'optG': optG_wg.state_dict(),
    'optC': optC_wg.state_dict(), 'g_losses': wgan_epoch_G, 'c_losses': wgan_epoch_C,
}, f'{CKPT_DIR}/wgan_full_checkpoint.pth')

# Config JSON for HuggingFace app
cfg = dict(
    NZ=NZ, NGF=NGF, NDF=NDF, NC=NC, IMAGE_SIZE=IMAGE_SIZE,
    DATASET=DATASET_CHOICE, DCGAN_EPOCHS=DCGAN_EPOCHS, WGAN_EPOCHS=WGAN_EPOCHS,
    dcgan_final_g_loss=round(float(dcgan_epoch_G[-1]), 4),
    dcgan_final_d_loss=round(float(dcgan_epoch_D[-1]), 4),
    wgan_final_g_loss =round(float(wgan_epoch_G[-1]),  4),
    wgan_final_c_loss =round(float(wgan_epoch_C[-1]),  4),
)
with open(f'{CKPT_DIR}/model_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

# Bundle everything into one ZIP for one-click download
ZIP = '/kaggle/working/gan_all_weights.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pth in Path(CKPT_DIR).rglob('*.pth'):
        zf.write(pth, f'weights/{pth.name}')
    zf.write(f'{CKPT_DIR}/model_config.json', 'weights/model_config.json')
    for png in sorted(Path(SAMPLE_DIR).rglob('*.png')):
        zf.write(png, f'samples/{png.relative_to(SAMPLE_DIR)}')

size_mb = Path(ZIP).stat().st_size / 1e6
print(f'ZIP ready: {ZIP}  ({size_mb:.1f} MB)')
print('Download from Kaggle OUTPUT tab (right panel).')
print()
print('Weight files:')
for f in sorted(Path(CKPT_DIR).glob('*.pth')):
    print(f'  {f.name:<45} {f.stat().st_size/1e6:.2f} MB')
print(f'  model_config.json')

ZIP ready: /kaggle/working/gan_all_weights.zip  (380.5 MB)
Download from Kaggle OUTPUT tab (right panel).

Weight files:
  dcgan_discriminator_final.pth                 11.08 MB
  dcgan_ep005.pth                               25.40 MB
  dcgan_ep010.pth                               25.40 MB
  dcgan_ep015.pth                               25.40 MB
  dcgan_ep017.pth                               25.40 MB
  dcgan_full_checkpoint.pth                     76.16 MB
  dcgan_generator_final.pth                     14.32 MB
  wgan_critic_final.pth                         11.07 MB
  wgan_ep005.pth                                25.39 MB
  wgan_ep010.pth                                25.39 MB
  wgan_ep015.pth                                25.39 MB
  wgan_ep017.pth                                25.39 MB
  wgan_full_checkpoint.pth                      76.15 MB
  wgan_generator_final.pth                      14.32 MB
  model_config.json


## Cell 19 - Generate HuggingFace app.py

In [19]:
APP_PY = '''import json, torch, gradio as gr, numpy as np
from PIL import Image
from torchvision.utils import make_grid
import torch.nn as nn

with open("model_config.json") as f:
    cfg = json.load(f)
NZ, NGF, NC = cfg["NZ"], cfg["NGF"], cfg["NC"]
DATASET = cfg["DATASET"]
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz,    ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z): return self.main(z)

g_dc = Generator(NZ, NGF, NC).to(device)
g_dc.load_state_dict(torch.load("dcgan_generator_final.pth", map_location=device))
g_dc.eval()

g_wg = Generator(NZ, NGF, NC).to(device)
g_wg.load_state_dict(torch.load("wgan_generator_final.pth", map_location=device))
g_wg.eval()

def gen(model_name, n, seed):
    torch.manual_seed(int(seed))
    noise = torch.randn(int(n), NZ, 1, 1, device=device)
    model = g_dc if model_name == "DCGAN" else g_wg
    with torch.no_grad():
        imgs = (model(noise) * 0.5 + 0.5).clamp(0, 1).cpu()
    grid = make_grid(imgs, nrow=min(int(n), 5), padding=2)
    return Image.fromarray((grid.permute(1,2,0).numpy()*255).astype("uint8"))

def compare(n, seed):
    return gen("DCGAN", n, seed), gen("WGAN-GP", n, seed)

with gr.Blocks(title=f"GAN - {DATASET.upper()}", theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# DCGAN vs WGAN-GP | {DATASET.upper()} | 64x64")
    with gr.Tab("Generate"):
        with gr.Row():
            md = gr.Dropdown(["DCGAN","WGAN-GP"], value="DCGAN", label="Model")
            ns = gr.Slider(1, 25, value=10, step=1, label="Images")
            si = gr.Number(value=42, label="Seed")
        out = gr.Image(type="pil", label="Output")
        gr.Button("Generate", variant="primary").click(gen, [md,ns,si], out)
    with gr.Tab("Compare Side-by-Side"):
        with gr.Row():
            cn = gr.Slider(1,10,value=5,step=1,label="Images per model")
            cs = gr.Number(value=42, label="Seed")
        od = gr.Image(label="DCGAN", type="pil")
        ow = gr.Image(label="WGAN-GP", type="pil")
        gr.Button("Compare", variant="primary").click(compare, [cn,cs], [od,ow])
    with gr.Tab("Model Info"):
        gr.Markdown(f"""| | DCGAN | WGAN-GP |
|---|---|---|
| G Loss | {cfg[\"dcgan_final_g_loss\"]} | {cfg[\"wgan_final_g_loss\"]} |
| D/C Loss | {cfg[\"dcgan_final_d_loss\"]} | {cfg[\"wgan_final_c_loss\"]} |
| Epochs | {cfg[\"DCGAN_EPOCHS\"]} | {cfg[\"WGAN_EPOCHS\"]} |""")
demo.launch()
'''

REQUIREMENTS = 'torch\ntorchvision\ngradio\nnumpy\nPillow\n'

with open('/kaggle/working/app.py', 'w') as f:           f.write(APP_PY)
with open('/kaggle/working/requirements.txt', 'w') as f: f.write(REQUIREMENTS)

for fname in ['dcgan_generator_final.pth', 'wgan_generator_final.pth', 'model_config.json']:
    src = Path(CKPT_DIR) / fname
    if src.exists():
        shutil.copy(src, f'/kaggle/working/{fname}')
        print(f'  Copied {fname}')

print()
print('HuggingFace Deployment - 4 steps:')
print('  1. Download gan_all_weights.zip from Kaggle OUTPUT tab')
print('  2. Go to huggingface.co/new-space  SDK=Gradio  Hardware=CPU')
print('  3. Upload: app.py  requirements.txt')
print('             dcgan_generator_final.pth')
print('             wgan_generator_final.pth')
print('             model_config.json')
print('  4. Space auto-builds - done!')

  Copied dcgan_generator_final.pth
  Copied wgan_generator_final.pth
  Copied model_config.json

HuggingFace Deployment - 4 steps:
  1. Download gan_all_weights.zip from Kaggle OUTPUT tab
  2. Go to huggingface.co/new-space  SDK=Gradio  Hardware=CPU
  3. Upload: app.py  requirements.txt
             dcgan_generator_final.pth
             wgan_generator_final.pth
             model_config.json
  4. Space auto-builds - done!


## Cell 20 - Live Gradio Demo

In [20]:
import gradio as gr

netG_dc.eval(); netG_wg.eval()

def generate_live(model_choice, n_images, seed):
    torch.manual_seed(int(seed))
    noise = torch.randn(int(n_images), NZ, 1, 1, device=device)
    model = netG_dc if model_choice == 'DCGAN' else netG_wg
    with torch.no_grad():
        imgs = (model(noise) * 0.5 + 0.5).clamp(0, 1).cpu()
    grid = make_grid(imgs, nrow=min(int(n_images), 5), padding=2)
    return Image.fromarray((grid.permute(1,2,0).numpy() * 255).astype(np.uint8))

def compare_live(n_images, seed):
    return generate_live('DCGAN', n_images, seed), generate_live('WGAN-GP', n_images, seed)

with gr.Blocks(title='DCGAN vs WGAN-GP', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# DCGAN vs WGAN-GP | {DATASET_CHOICE.upper()} Dataset | 64x64')

    with gr.Tab('Single Model'):
        with gr.Row():
            m_dd = gr.Dropdown(['DCGAN','WGAN-GP'], value='DCGAN', label='Model')
            n_sl = gr.Slider(1, 25, value=10, step=1, label='# Images')
            s_in = gr.Number(value=42, label='Seed')
        out_img = gr.Image(label='Generated', type='pil')
        gr.Button('Generate', variant='primary').click(
            generate_live, [m_dd, n_sl, s_in], out_img)

    with gr.Tab('Compare Both Models'):
        with gr.Row():
            c_n = gr.Slider(1, 10, value=5, step=1, label='Images per model')
            c_s = gr.Number(value=42, label='Seed')
        with gr.Row():
            dc_out = gr.Image(label='DCGAN Output',   type='pil')
            wg_out = gr.Image(label='WGAN-GP Output', type='pil')
        gr.Button('Compare', variant='primary').click(
            compare_live, [c_n, c_s], [dc_out, wg_out])

    with gr.Tab('Training Summary'):
        gr.Markdown(f"""
| Metric | DCGAN | WGAN-GP |
|--------|-------|---------|
| Loss | BCEWithLogitsLoss | Wasserstein + GP |
| D output | Raw logit | Raw score |
| Normalisation | BatchNorm | InstanceNorm |
| Mode collapse | Prone | Resolved |
| Critic steps/G | 1 | {N_CRITIC} |
| Lambda GP | - | {LAMBDA_GP} |
| Epochs | {DCGAN_EPOCHS} | {WGAN_EPOCHS} |
| Final G Loss | {dcgan_epoch_G[-1]:.4f} | {wgan_epoch_G[-1]:.4f} |
        """)

demo.launch(share=True, debug=False)
print('Gradio app launched!')

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bea350c074179ba64a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio app launched!


---
## Summary

| Aspect | DCGAN | WGAN-GP |
|--------|-------|---------|
| Loss | BCEWithLogitsLoss (AMP-safe) | Wasserstein + Gradient Penalty |
| D/C output | Raw logit | Raw score |
| Normalisation | BatchNorm | InstanceNorm |
| Mode collapse | Prone | Eliminated |
| Epochs | 17 | 17 |
| Visualised at | Epoch 5, 10, 15, 17 | Epoch 5, 10, 15, 17 |
| Checkpoints | Epoch 5, 10, 15, 17 | Epoch 5, 10, 15, 17 |

**Download**: `gan_all_weights.zip` from Kaggle OUTPUT tab.